# ROOT Python-C++ interoperability

ROOT is a powerful framework that allows blending C++ and Python functionality. It relies on cling for C++ interpretation and it implements dynamic Python bindings. That means that ROOT functionality written in C++ is not exposed statically in Python by writing extra boilerplate code to expose the same function to the Python interpreter. Instead, C++ code can be exposed directly for use in a Python session. In this notebook, we will take a look at various examples that show the breadth of use cases that such an engine can support.

## Exposing C++ functions to Python

Some parts of your application may benefit from delegating to C++ code the more heavy-duty computations or from thread-based parallelism which is more easily achieved in C++ than in Python. With ROOT, this becomes trivial.

In [ ]:
import ROOT

Let's define a C++ function. Note that we use the `%%cpp` magic in the following notebook cell, which will provide a true C++ coding environment, with proper syntax highlighting. Thys magic keyword is provided in a Jupyter notebook after `import ROOT`

In [ ]:
%%cpp --declare

#ifndef MYUTILS
#define MYUTILS

#include <algorithm>
#include <vector>

float sum_v(const std::vector<float> &vec){
    return std::accumulate(vec.begin(), vec.end(), 0.f);
}

#endif // MYUTILS

The function can now be seamlessly used in a Python application:

In [ ]:
import numpy

arr = numpy.array([1,2,3,4,5], dtype=numpy.float32)
s = ROOT.sum_v(arr)
print(f"Sum of values: {s}")

Note the interaction between a C++ function and a Python class such as `numpy.array`. In this case, there is a hidden logic that exposes the numpy array buffer as a view to the `std::vector<float>` parameter of the `sum_v` function. Also note that the `numpy.array` was created with `dtype=numpy.float32` explicitly to ensure the same underlying value type is used.

The following code will do the equivalent of the cell above where we defined the function via the `%%cpp --declare` magic and it can be used in any Python script:

```python
ROOT.gInterpreter.Declare("""
#ifndef MYUTILS
#define MYUTILS

#include <algorithm>
#include <vector>

float sum_v(const std::vector<float> &vec){
    return std::accumulate(vec.begin(), vec.end(), 0.f);
}

#endif // MYUTILS
""")
```

## Using objects created in C++ inside of a Python function

Another generic type of use case is creating a C++ object holding some data, then use it on the Python side. Take a look at the following:

In [ ]:
%%cpp --declare
#ifndef MYSTRUCT
#define MYSTRUCT

#include <vector>

struct DataHolder{
    std::vector<int> data{1, 2, 3, 4, 5};
};

#endif

In [ ]:
%%cpp

DataHolder mydata;

In [ ]:
def process_data(data_holder):
    return numpy.sum(data_holder.data)

print(f"{process_data(ROOT.mydata)=}")

### An important note

In this example, we have used the `%%cpp` magic twice. The first time, `%%cpp --declare` was used to declare the struct holding the data. The second time, `%%cpp` was used to create an instance of the struct in memory. Finally, in the Python cell we could access that C++ object instance as `ROOT.mydata` from the Python side. The equivalent code for the `%%cpp` magic cells that could be executed in any Python script is:

```python
ROOT.gInterpreter.Declare("""
#ifndef MYSTRUCT
#define MYSTRUCT

#include <vector>

struct DataHolder{
    std::vector<int> data{1, 2, 3, 4, 5};
};

#endif
""")
ROOT.gInterpreter.ProcessLine("""
DataHolder mydata;
""")
```

Here we see a key difference between the `Declare` and `ProcessLine` methods of the C++ interpreter. The first one is used to declare code that the rest of the application should be able to access: function, classes, namespaces etc. The second one is used to actually execute C++ statements, thus can be useful to generate objects that can be later accessed on the Python side.

## ROOT Pythonizations

The possibilities enabled by the dynamic Python bindings don't just stop at exposing C++ code to Python. Another widely-scoped functionality is the ability to produce so-called **"Pythonizations"**. With this term, we refer to logic written in Python that changes the behaviour of a certain C++ class when it is used within a Python application. For example:

In [ ]:
ROOT.std.vector["float"](numpy.random.random_sample(5))

In this example, the `std::vector<float>` C++ is enhanced on the Python side with the ability to pretty-print its contents with the format `vector<TYPE>{ VALUES... }`, with a similar behaviour to a Python `list`. While this is done automatically in ROOT, it can be done for any custom C++ class, including yours!

Let's begin by creating a class that represents a table with two columns, `x` and `y`. The values of the columns can be provided by external users of the class:

In [ ]:
%%cpp --declare

#ifndef SIMPLETABLE
#define SIMPLETABLE

#include <vector>

class SimpleTable{

    std::vector<int> fX;
    std::vector<int> fY;

public:
    SimpleTable(const std::vector<int> &x, const std::vector<int> &y): fX(x), fY(y) {}

    const std::vector<int> &X() const { return fX; }
    const std::vector<int> &Y() const { return fY; }
};

#endif

In [ ]:
from ROOT import pythonization

def convert_to_markdown(simpletable):

    markdown_table = "|  x  |  y  |\n|-----|-----|\n"    
    for x, y in zip(simpletable.X(), simpletable.Y()):
        markdown_table += "|   " + str(x) + " |  " + str(y) + " |\n"

    return markdown_table


@pythonization("SimpleTable")
def _(klass):
    klass.__repr__ = lambda self: convert_to_markdown(self)

In [ ]:
ROOT.SimpleTable([1,2,3,4,5], [42,43,44,45,46])

In the example above, the C++ class was only holding the data. On the python side, we have created a function to convert the data to a simple markdown table and then we have injected it as a custom behaviour of the class via the `__repr__` method, which is used in the Jupyter notebook cell to display a representation of the object.